# Comprehensive Build Trace Analysis Guide

This notebook demonstrates how to analyze C++ build performance using Clang's `-ftime-trace` feature. We'll explore the trace analysis library and show practical techniques for understanding and improving compilation times.

## The Problem: C++ Metaprogramming Build Times

The Composable Kernel (CK) library uses extensive C++17 metaprogramming to generate high-performance GPU kernels. While this approach provides excellent runtime performance, it comes with a cost: **long compilation times**.

Understanding where the compiler spends its time is critical for:

- **Identifying bottlenecks**: Which templates are most expensive?
- **Measuring progress**: Are our optimizations working?
- **Focusing efforts**: Where should we invest time to improve build performance?

## The Solution: Data-Driven Analysis

Clang's `-ftime-trace` flag generates detailed JSON files showing exactly where compilation time is spent. This notebook shows how to:

1. **Parse** trace files efficiently using parallel processing
2. **Transform** raw JSON into structured multi-table schemas
3. **Analyze** build performance with relational queries
4. **Visualize** build parallelism with Gantt charts
5. **Identify** optimization opportunities

Let's treat this as a **big data problem** and use the best tools available: pandas, parallel processing, and Jupyter notebooks.

## Setup and Imports

In [1]:
from importlib.util import find_spec
from multiprocessing import cpu_count
import pandas as pd
from pathlib import Path
import sys
import time

# Add parent directory to path to import trace_analysis
sys.path.insert(0, str(Path.cwd().parent))

from trace_analysis import TraceFile, TraceParser, TraceTransformer, find_trace_files

# Check for optional dependencies
HAS_TQDM = find_spec("tqdm") is not None
if not HAS_TQDM:
    print("Note: Install tqdm for progress bars: pip install tqdm")

HAS_PLOTLY = find_spec("plotly") is not None
if not HAS_PLOTLY:
    print("Note: Install plotly for visualizations: pip install plotly")

# Display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 80)

print(f"Using {cpu_count()} CPU cores for parallel processing")
print(f"Pandas version: {pd.__version__}")

Note: Install plotly for visualizations: pip install plotly
Using 384 CPU cores for parallel processing
Pandas version: 2.3.3


## Part 1: Single File Analysis

Let's start by analyzing a single trace file to understand the schema structure.

We first identify all the json files from a build with `-ftime-trace` set in the `CXX_FLAGS`.

In [2]:
# Configure the path to your trace files
TRACE_DIR = Path("../../../build-trace")

sample_files = find_trace_files(TRACE_DIR)

if not sample_files:
    print(f"No trace files found in {TRACE_DIR}")
    print("\nTo generate trace files:")
    print("1. Configure your build with: cmake -DCMAKE_CXX_FLAGS='-ftime-trace' ...")
    print("2. Build your project")
    print("3. Trace files will be generated alongside object files")
else:
    print(f"Found {len(sample_files):,} trace files")
    sample_file = sample_files[0]
    print(f"\nUsing sample file: {sample_file.name}")
    print(f"File size: {sample_file.stat().st_size / 1024:.1f} KB")

Found 1,279 trace files

Using sample file: device_conv1d_bwd_data_xdl_nwc_kxc_nwk_int8_instance.cpp.json
File size: 10967.8 KB


### Parsing to Pandas Data Frames

The Pandas schema creates separate DataFrames for:
- **templates**: Unique template definitions with structure
- **instantiations**: Template instantiation events with timing
- **template_args**: Template argument relationships
- **events**: All compiler events

This normalized structure enables efficient queries and reduces memory usage.

In [3]:
if sample_files:
    # Parse the trace file
    trace_file = TraceFile.from_path(sample_file)

    start = time.time()
    events = TraceParser.parse(trace_file)
    parse_time = time.time() - start

    # Get beginning of time for timeline analysis
    import orjson

    with open(sample_file, "rb") as f:
        trace_data = orjson.loads(f.read())
    beginning_of_time = TraceTransformer.extract_beginning_of_time(trace_data)

    # Convert to enhanced schema
    start = time.time()
    tables = TraceTransformer.to_enhanced_schema(
        events, file_id=0, beginning_of_time_us=beginning_of_time
    )
    transform_time = time.time() - start

    print(f"Parsed {len(events):,} events in {parse_time:.3f}s")
    print(f"Transformed to Pandas tables in {transform_time:.3f}s\n")

    print("Pandas DataFrames:")
    for name, df in tables.items():
        mem_mb = df.memory_usage(deep=True).sum() / 1024**2
        cols = ", ".join(df.columns)
        print(f"  {name:20s}: {len(df):6,} rows, {mem_mb:6.2f} MB | {cols}")

Parsed 14,271 events in 0.058s
Transformed to Pandas tables in 0.999s

Pandas DataFrames:
  templates           :  8,156 rows,   7.56 MB | template_id, template_name, full_signature, depth, arg_count
  instantiations      :  9,267 rows,   0.29 MB | instantiation_id, template_id, file_id, dur_us, ts_us, event_type
  template_args       : 49,097 rows,   8.23 MB | parent_template_id, arg_position, arg_template_id, arg_type, arg_text
  events              : 14,271 rows,   0.50 MB | name, dur, ts, pid, tid, ph, ts_absolute_us


Summarize the compilation unit (start time, duration, and any other summary statistics)

In [4]:
if sample_files:
    print("Compilation Unit Summary:")
    print(f"  Trace file: {sample_file.name}")
    print(f"  Trace file size: {sample_file.stat().st_size / 1024:.1f} KB")
    print(f"  Start time: {pd.to_datetime(beginning_of_time, unit='us')}")
    print(f"  Total compilation time: {tables['events']['dur'].sum() / 1e6:.2f}s")
    print(f"  Total events: {len(tables['events']):,}")

Compilation Unit Summary:
  Trace file: device_conv1d_bwd_data_xdl_nwc_kxc_nwk_int8_instance.cpp.json
  Trace file size: 10967.8 KB
  Start time: 2026-01-04 16:14:25.048921
  Total compilation time: 162.17s
  Total events: 14,271


### Examining the Templates Table

The templates table contains unique template definitions with parsed structure.

In [5]:
if sample_files:
    templates_df = tables["templates"]

    # Filter to CK templates only (exclude std library templates)
    ck_templates_df = templates_df[
        templates_df["template_name"].str.startswith("ck::", na=False)
        | templates_df["template_name"].str.startswith("ck_tile::", na=False)
    ]

    print("Templates DataFrame Schema:")

    # Combine dtype and memory info
    mem_usage = templates_df.memory_usage(deep=True)
    total_mb = mem_usage.sum() / 1024**2

    print(f"{'Column':<25s} {'Type':<15s} {'Memory (MB)':>12s} {'% of Total':>12s}")
    print("-" * 67)
    for col in templates_df.columns:
        dtype_str = str(templates_df[col].dtype)
        mem_mb = mem_usage[col] / 1024**2
        pct = 100 * mem_usage[col] / mem_usage.sum()
        print(f"{col:<25s} {dtype_str:<15s} {mem_mb:12.2f} {pct:11.1f}%")

    # Add Index row
    idx_mem_mb = mem_usage["Index"] / 1024**2
    idx_pct = 100 * mem_usage["Index"] / mem_usage.sum()
    print(f"{'Index':<25s} {'RangeIndex':<15s} {idx_mem_mb:12.2f} {idx_pct:11.1f}%")
    print("-" * 67)
    print(f"{'TOTAL':<25s} {'':<15s} {total_mb:12.2f} {100.0:11.1f}%")

    print(f"\nTotal templates: {len(templates_df):,}")
    print(
        f"CK templates: {len(ck_templates_df):,} ({100 * len(ck_templates_df) / len(templates_df):.1f}%)"
    )
    print(f"Other templates: {len(templates_df) - len(ck_templates_df):,}")
    print("\nSample CK templates:")
    display(ck_templates_df.head(10))

Templates DataFrame Schema:
Column                    Type             Memory (MB)   % of Total
-------------------------------------------------------------------
template_id               int32                   0.03         0.4%
template_name             category                0.03         0.4%
full_signature            object                  7.49        99.0%
depth                     int8                    0.01         0.1%
arg_count                 int8                    0.01         0.1%
Index                     RangeIndex              0.00         0.0%
-------------------------------------------------------------------
TOTAL                                             7.56       100.0%

Total templates: 8,156
CK templates: 6,063 (74.3%)
Other templates: 2,093

Sample CK templates:


,template_id,template_name,full_signature,depth,arg_count
46,46,ck::vector_type,"ck::vector_type<float, 2>::(unnamed union at /home/AMD/jshumway/composable_k...",1,2
47,47,ck::vector_type,"ck::vector_type<float, 2>",1,2
48,48,ck::vector_type,"ck::vector_type<float, 4>::(unnamed union at /home/AMD/jshumway/composable_k...",1,2
49,49,ck::vector_type,"ck::vector_type<float, 4>",1,2
50,50,ck::detail::TupleElementKeyData,"ck::detail::TupleElementKeyData<ck::detail::TupleElementKey<3>, float __attr...",2,2
51,51,ck::detail::TupleImpl,"ck::detail::TupleImpl<ck::Sequence<0, 1, 2, 3>, float __attribute__((ext_vec...",2,5
52,52,ck::Tuple,"ck::Tuple<float __attribute__((ext_vector_type(2))), float __attribute__((ex...",1,4
53,53,ck::vector_type,"ck::vector_type<float, 8>::(unnamed union at /home/AMD/jshumway/composable_k...",1,2
54,54,ck::vector_type,"ck::vector_type<float, 8>",1,2
55,55,ck::Tuple,"ck::Tuple<float, float, float, float, float, float, float, float, float, flo...",1,16


### Template Instantiation Analysis

Join templates with instantiations to analyze performance.

In [6]:
if sample_files and len(tables["instantiations"]) > 0:
    # Join instantiations with templates
    inst_df = tables["instantiations"].merge(
        tables["templates"][
            ["template_id", "template_name", "full_signature", "depth"]
        ],
        on="template_id",
    )

    total_template_time_us = inst_df["dur_us"].sum()
    total_time_us = tables["events"]["dur"].sum()

    print("Template Instantiation Summary:")
    print(f"  Unique templates: {len(tables['templates']):,}")
    print(f"  Total instantiations: {len(inst_df):,}")
    print(f"  Template time: {total_template_time_us / 1e6:.2f}s")
    print(f"  Percentage of build: {100 * total_template_time_us / total_time_us:.1f}%")
    print(f"  Avg per instantiation: {inst_df['dur_us'].mean() / 1e3:.2f} ms")

    # Most expensive templates by total time
    print("\nTop 10 Templates by Total Time:")
    template_stats = (
        inst_df.groupby("template_id")
        .agg(
            {
                "dur_us": ["count", "sum", "mean", "max"],
                "full_signature": "first",
                "depth": "first",
            }
        )
        .reset_index()
    )
    template_stats.columns = [
        "template_id",
        "count",
        "total_us",
        "mean_us",
        "max_us",
        "signature",
        "depth",
    ]
    template_stats["total_ms"] = template_stats["total_us"] / 1e3
    template_stats["mean_ms"] = template_stats["mean_us"] / 1e3

    display(
        template_stats.nlargest(10, "total_us")[
            ["signature", "count", "total_ms", "mean_ms", "depth"]
        ]
    )

Template Instantiation Summary:
  Unique templates: 8,156
  Total instantiations: 9,267
  Template time: 36.26s
  Percentage of build: 22.4%
  Avg per instantiation: 3.91 ms

Top 10 Templates by Total Time:


,signature,count,total_ms,mean_ms,depth
620,ck::generate_tuple<(lambda at /home/AMD/jshumway/composable_kernel/include/c...,12,403.014,33.584500,1
619,ck::generate_tuple_for<(lambda at /home/AMD/jshumway/composable_kernel/inclu...,12,401.766,33.480500,1
1305,"ck::detail::applier<int, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10>::operator()<(lamb...",34,330.576,9.722824,1
6125,"ck::detail::applier<int, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12>::operator...",2,318.451,159.225500,1
3881,ck::generate_tuple<(lambda at /home/AMD/jshumway/composable_kernel/include/c...,4,303.042,75.760500,1
3880,ck::generate_tuple_for<(lambda at /home/AMD/jshumway/composable_kernel/inclu...,4,302.517,75.629250,1
4631,ck::transform_tensor_descriptor<ck::TensorDescriptor<ck::Tuple<ck::UnMerge<c...,1,301.674,301.674000,6
2217,ck::generate_tuple<(lambda at /home/AMD/jshumway/composable_kernel/include/c...,5,290.939,58.187800,1
2216,ck::generate_tuple_for<(lambda at /home/AMD/jshumway/composable_kernel/inclu...,5,290.336,58.067200,1
1364,ck::generate_tuple<(lambda at /home/AMD/jshumway/composable_kernel/include/c...,6,284.086,47.347667,1


### Template Depth Analysis

Analyze template nesting depth to identify complex metaprogramming patterns.

In [7]:
if sample_files and len(tables["templates"]) > 0:
    depth_stats = (
        inst_df.groupby("depth").agg({"dur_us": ["count", "sum", "mean"]}).reset_index()
    )
    depth_stats.columns = ["depth", "count", "total_us", "mean_us"]
    depth_stats["total_ms"] = depth_stats["total_us"] / 1e3
    depth_stats["mean_ms"] = depth_stats["mean_us"] / 1e3

    print("Template Instantiation by Nesting Depth:")
    display(depth_stats[["depth", "count", "total_ms", "mean_ms"]])

Template Instantiation by Nesting Depth:


,depth,count,total_ms,mean_ms
0,1,1564,7735.584,4.946026
1,2,1873,4943.003,2.639083
2,3,1256,4686.911,3.731617
3,4,2336,6986.861,2.990951
4,5,1969,9640.982,4.896385
5,6,252,2251.077,8.932845
6,7,17,18.260,1.074118


## Part 2: Multi-File Analysis


⚠️ **Warning** The timings looks suspcious on some of these joined tables. We need to do more testing and development.


Now let's scale up to analyze an entire build using parallel processing.

In [8]:
from concurrent.futures import ProcessPoolExecutor, as_completed


def process_file(json_path: Path, file_id: int) -> dict:
    """Process a single trace file and return enhanced schema tables."""
    import orjson
    from trace_analysis import TraceFile, TraceParser, TraceTransformer

    trace_file = TraceFile.from_path(json_path)
    events = TraceParser.parse(trace_file)

    # Extract beginning of time
    with open(json_path, "rb") as f:
        trace_data = orjson.loads(f.read())
    beginning_of_time = TraceTransformer.extract_beginning_of_time(trace_data)

    # Convert to Pandas DataFrames
    tables = TraceTransformer.to_enhanced_schema(
        events, file_id=file_id, beginning_of_time_us=beginning_of_time
    )

    return {"file_id": file_id, "file_name": json_path.name, "tables": tables}


print("Parallel processing function defined")

Parallel processing function defined


In [9]:
# Find all trace files
json_files = find_trace_files(TRACE_DIR)

if not json_files:
    print(f"No trace files found in {TRACE_DIR}")
else:
    print(f"Found {len(json_files):,} trace files")
    total_size = sum(f.stat().st_size for f in json_files)
    print(f"Total size: {total_size / 1024**3:.2f} GB")

    # For demonstration, you might want to limit the number of files
    # Uncomment the next line to process only the first 100 files
    # json_files = json_files[:100]

Found 1,279 trace files
Total size: 17.20 GB


### Processing All Files in Parallel

In [10]:
if json_files:
    print(f"Processing {len(json_files):,} files with {cpu_count()} workers...\n")

    start_time = time.time()
    results = []

    # Submit all files for parallel processing
    with ProcessPoolExecutor(max_workers=cpu_count()) as executor:
        futures = {
            executor.submit(process_file, f, i): (f, i)
            for i, f in enumerate(json_files)
        }

        # Collect results with progress bar
        if HAS_TQDM:
            from tqdm.auto import tqdm

            pbar = tqdm(total=len(json_files), desc="Processing", unit="files")

        for future in as_completed(futures):
            result = future.result()
            results.append(result)

            if HAS_TQDM:
                pbar.update(1)

        if HAS_TQDM:
            pbar.close()

    parse_time = time.time() - start_time
    print(
        f"\nParsing complete in {parse_time:.2f}s ({len(json_files) / parse_time:.1f} files/sec)"
    )

    # Combine all tables
    print("\nCombining results...")
    combine_start = time.time()

    all_templates = []
    all_instantiations = []
    all_template_args = []
    all_events = []

    for result in results:
        tables = result["tables"]
        if len(tables["templates"]) > 0:
            all_templates.append(tables["templates"])
        if len(tables["instantiations"]) > 0:
            all_instantiations.append(tables["instantiations"])
        if len(tables["template_args"]) > 0:
            all_template_args.append(tables["template_args"])
        if len(tables["events"]) > 0:
            all_events.append(tables["events"])

    # Concatenate DataFrames
    templates_df = (
        pd.concat(all_templates, ignore_index=True) if all_templates else pd.DataFrame()
    )
    instantiations_df = (
        pd.concat(all_instantiations, ignore_index=True)
        if all_instantiations
        else pd.DataFrame()
    )
    template_args_df = (
        pd.concat(all_template_args, ignore_index=True)
        if all_template_args
        else pd.DataFrame()
    )
    events_df = (
        pd.concat(all_events, ignore_index=True) if all_events else pd.DataFrame()
    )

    combine_time = time.time() - combine_start
    total_time = time.time() - start_time

    print(f"Combined in {combine_time:.2f}s")
    print(f"\nTotal analysis time: {total_time:.2f}s")

    # Calculate memory usage
    total_memory = (
        templates_df.memory_usage(deep=True).sum()
        + instantiations_df.memory_usage(deep=True).sum()
        + template_args_df.memory_usage(deep=True).sum()
        + events_df.memory_usage(deep=True).sum()
    ) / 1024**3

    print("\nCombined Tables:")
    print(f"  Templates: {len(templates_df):,} rows")
    print(f"  Instantiations: {len(instantiations_df):,} rows")
    print(f"  Template Args: {len(template_args_df):,} rows")
    print(f"  Events: {len(events_df):,} rows")
    print(f"  Total memory: {total_memory:.2f} GB")

Processing 1,279 files with 384 workers...



/home/AMD/jshumway/composable_kernel/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing: 100%|██████████| 1279/1279 [00:45<00:00, 28.08files/s] 



Parsing complete in 54.47s (23.5 files/sec)

Combining results...
Combined in 3.85s

Total analysis time: 58.32s

Combined Tables:
  Templates: 10,655,045 rows
  Instantiations: 15,806,599 rows
  Template Args: 144,341,805 rows
  Events: 22,196,001 rows
  Total memory: 31.69 GB


### Build-Wide Statistics

In [11]:
if json_files and len(events_df) > 0:
    total_build_time_us = events_df["dur"].sum()
    total_template_time_us = instantiations_df["dur_us"].sum()

    print("=" * 80)
    print("BUILD-WIDE STATISTICS")
    print("=" * 80)
    print(f"Files processed: {len(json_files):,}")
    print(f"Total events: {len(events_df):,}")
    print(f"Total build time: {total_build_time_us / 1e6 / 60:.2f} minutes")
    print(f"Unique templates: {len(templates_df):,}")
    print(f"Template instantiations: {len(instantiations_df):,}")
    print(
        f"Template time: {total_template_time_us / 1e6 / 60:.2f} minutes ({100 * total_template_time_us / total_build_time_us:.1f}%)"
    )
    print("=" * 80)

BUILD-WIDE STATISTICS
Files processed: 1,279
Total events: 22,196,001
Total build time: 6935.67 minutes
Unique templates: 10,655,045
Template instantiations: 15,806,599
Template time: 2007.82 minutes (28.9%)


### Top Templates by Total Time

Aggregate instantiations first, then join with templates for optimal performance.

⚠️ **Warning** Many of these times are identical. The join may be incorrect.

In [12]:
if json_files and len(instantiations_df) > 0:
    # OPTIMIZATION: Aggregate FIRST, then join (much faster!)
    # This reduces 27M rows to ~20M unique templates before joining
    print("Aggregating template statistics...")
    start = time.time()

    template_stats = (
        instantiations_df.groupby("template_id")
        .agg({"dur_us": ["count", "sum", "mean", "median", "max"]})
        .reset_index()
    )

    template_stats.columns = [
        "template_id",
        "count",
        "total_us",
        "mean_us",
        "median_us",
        "max_us",
    ]

    # Now join with templates (much smaller aggregated dataset)
    template_stats = template_stats.merge(
        templates_df[
            ["template_id", "template_name", "full_signature", "depth", "arg_count"]
        ],
        on="template_id",
    )

    # Add computed columns
    template_stats["total_s"] = template_stats["total_us"] / 1e6
    template_stats["mean_ms"] = template_stats["mean_us"] / 1e3
    template_stats["median_ms"] = template_stats["median_us"] / 1e3
    template_stats["pct_template_time"] = (
        100 * template_stats["total_us"] / total_template_time_us
    )

    elapsed = time.time() - start
    print(f"Completed in {elapsed:.2f}s\n")

    print("Top 20 Templates by Total Time:")
    display(
        template_stats.nlargest(20, "total_us")[
            [
                "full_signature",
                "count",
                "total_s",
                "mean_ms",
                "median_ms",
                "depth",
                "pct_template_time",
            ]
        ]
    )

Aggregating template statistics...
Completed in 3.10s

Top 20 Templates by Total Time:


,full_signature,count,total_s,mean_ms,median_ms,depth,pct_template_time
7340718,std::__uniq_ptr_impl<ck::tensor_operation::device::DeviceConv2dBwdDataXdl_In...,984,105.25365,106.965091,2.2285,4,0.08737
7340719,"ck::ThreadwiseTensorSliceTransfer_v3r2<const ck::Sequence<2, 2>, ck::tensor_...",984,105.25365,106.965091,2.2285,6,0.08737
7340720,"std::_TupleConstraints<true, ck::tensor_operation::device::DeviceConv2dBwdDa...",984,105.25365,106.965091,2.2285,4,0.08737
7340721,ck::kernel_batched_elementwise<ck::GridwiseElementwise<ck::Tuple<ck::TensorD...,984,105.25365,106.965091,2.2285,8,0.08737
7340722,std::tuple<ck::tensor_operation::device::DeviceConv2dBwdDataXdl_Input_N_Hi_W...,984,105.25365,106.965091,2.2285,4,0.08737
7340723,std::__uniq_ptr_data<ck::tensor_operation::device::DeviceConv2dBwdDataXdl_In...,984,105.25365,106.965091,2.2285,4,0.08737
7340724,std::__and_<std::__and_<std::is_constructible<ck::tensor_operation::device::...,984,105.25365,106.965091,2.2285,6,0.08737
7340725,"ck::GridwiseGemm_xdl_cshuffle_conv_v3<ck::tensor_layout::gemm::RowMajor, ck:...",984,105.25365,106.965091,2.2285,6,0.08737
7340726,"ck::detail::applier<int, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...",984,105.25365,106.965091,2.2285,1,0.08737
7340727,ck::make_tensor_coordinate(const ck::TensorDescriptor<ck::Tuple<ck::Embed<ck...,984,105.25365,106.965091,2.2285,5,0.08737


### Filter to CK Namespaces Only

Filter template statistics to show only `ck::` and `ck_tile::` namespaces, excluding standard library templates.

In [13]:
if json_files and len(template_stats) > 0:
    # Filter to only CK namespaces
    ck_template_stats = template_stats[
        template_stats["template_name"].str.startswith("ck::", na=False)
        | template_stats["template_name"].str.startswith("ck_tile::", na=False)
    ].copy()

    # Recalculate percentage based on filtered CK templates only
    ck_total_time_us = ck_template_stats["total_us"].sum()
    ck_template_stats["pct_ck_time"] = (
        100 * ck_template_stats["total_us"] / ck_total_time_us
    )

    print(
        f"Filtered to {len(ck_template_stats):,} CK templates (from {len(template_stats):,} total)"
    )
    print(f"CK template time: {ck_total_time_us / 1e6:.2f}s")
    print(
        f"Percentage of total template time: {100 * ck_total_time_us / total_template_time_us:.1f}%"
    )

    print("\nTop 20 CK Templates by Total Time:")
    display(
        ck_template_stats.nlargest(20, "total_us")[
            [
                "full_signature",
                "count",
                "total_s",
                "mean_ms",
                "median_ms",
                "depth",
                "pct_ck_time",
            ]
        ]
    )

Filtered to 8,644,580 CK templates (from 10,655,045 total)
CK template time: 54849416.17s
Percentage of total template time: 45529.9%

Top 20 CK Templates by Total Time:


,full_signature,count,total_s,mean_ms,median_ms,depth,pct_ck_time
7340719,"ck::ThreadwiseTensorSliceTransfer_v3r2<const ck::Sequence<2, 2>, ck::tensor_...",984,105.25365,106.965091,2.2285,6,0.000192
7340721,ck::kernel_batched_elementwise<ck::GridwiseElementwise<ck::Tuple<ck::TensorD...,984,105.25365,106.965091,2.2285,8,0.000192
7340725,"ck::GridwiseGemm_xdl_cshuffle_conv_v3<ck::tensor_layout::gemm::RowMajor, ck:...",984,105.25365,106.965091,2.2285,6,0.000192
7340726,"ck::detail::applier<int, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...",984,105.25365,106.965091,2.2285,1,0.000192
7340727,ck::make_tensor_coordinate(const ck::TensorDescriptor<ck::Tuple<ck::Embed<ck...,984,105.25365,106.965091,2.2285,5,0.000192
7340728,"ck::TensorDescriptor<ck::Tuple<ck::Embed<ck::Tuple<int, int, int, int, int>,...",984,105.25365,106.965091,2.2285,5,0.000192
7340729,ck::TensorDescriptor<ck::Tuple<ck::UnMerge<ck::Tuple<ck::integral_constant<i...,984,105.25365,106.965091,2.2285,5,0.000192
7340730,"ck::TensorDescriptor<ck::Tuple<ck::Embed<ck::Tuple<int, int, int, int, int>,...",984,105.25365,106.965091,2.2285,5,0.000192
7340731,"ck::StaticTensorTupleOfVectorBuffer<ck::AddressSpaceEnum::Vgpr, _Float16, 8,...",984,105.25365,106.965091,2.2285,6,0.000192
7340732,ck::make_tensor_coordinate_step(const ck::TensorDescriptor<ck::Tuple<ck::Emb...,984,105.25365,106.965091,2.2285,5,0.000192


### Most Frequently Instantiated Templates

In [14]:
if json_files and len(template_stats) > 0:
    print("Top 20 Most Frequently Instantiated Templates:")
    display(
        template_stats.nlargest(20, "count")[
            ["full_signature", "count", "total_s", "mean_ms", "depth"]
        ]
    )

Top 20 Most Frequently Instantiated Templates:


,full_signature,count,total_s,mean_ms,depth
6304619,"ck_tile::impl::run_cast_from_f8<_BitInt(8), float, true>",9773,23.119388,2.365639,1
6304620,"ck::ThreadwiseTensorSliceTransfer_v3r2<const ck::Sequence<4, 16>, ck::tensor...",9773,23.119388,2.365639,6
6304621,"ck::detail::TupleImpl<ck::Sequence<0, 1, 2, 3, 4, 5, 6, 7>, ck::Sequence<19>...",9773,23.119388,2.365639,2
6304622,"ck::operator<<(std::ostream &, const TensorDescriptor<Tuple<Embed<Tuple<int,...",9773,23.119388,2.365639,7
6304623,ck::TensorDescriptor<ck::Tuple<ck::Embed<ck::Tuple<ck::integral_constant<int...,9773,23.119388,2.365639,5
6304624,"ck::sequence_gen<8, ck::lambda_get_up_dim_num<ck::Tuple<ck::PassThrough<int>...",9773,23.119388,2.365639,4
6304625,std::unique_ptr<ck::tensor_operation::device::DeviceGroupedConvFwdMultipleAB...,9773,23.119388,2.365639,3
6304626,"ck::container_concat<ck::Sequence<0>, ck::Sequence<2, 1>, ck::Sequence<3>, c...",9773,23.119388,2.365639,2
6304627,"ck::Sequence<19, 20, 21, 22, 23>::ReorderGivenNew2Old<0, 1, 2, 3, 4>",9773,23.119388,2.365639,1
6304628,"ck::make_tuple<ck::Sequence<0>, ck::Sequence<>, ck::Sequence<1>, ck::Sequenc...",9773,23.119388,2.365639,2


## Part 3: Advanced Analysis

### Optimization Priority Score

Templates that are both frequently instantiated AND expensive per instantiation are prime optimization targets.

⚠️ **Warning** Many of these times are identical, `template_stats` may be incorrct.


In [15]:
if json_files and len(template_stats) > 0:
    # Normalize count and mean to 0-1 range
    template_stats["count_norm"] = (
        template_stats["count"] - template_stats["count"].min()
    ) / (template_stats["count"].max() - template_stats["count"].min())
    template_stats["mean_norm"] = (
        template_stats["mean_us"] - template_stats["mean_us"].min()
    ) / (template_stats["mean_us"].max() - template_stats["mean_us"].min())

    # Priority score: weighted combination
    template_stats["priority_score"] = (
        0.5 * template_stats["count_norm"] + 0.5 * template_stats["mean_norm"]
    )

    print("Top 15 Optimization Targets (High Frequency + High Cost):")
    display(
        template_stats.nlargest(15, "priority_score")[
            ["full_signature", "count", "total_s", "mean_ms", "priority_score"]
        ]
    )

Top 15 Optimization Targets (High Frequency + High Cost):


,full_signature,count,total_s,mean_ms,priority_score
6304619,"ck_tile::impl::run_cast_from_f8<_BitInt(8), float, true>",9773,23.119388,2.365639,0.500023
6304620,"ck::ThreadwiseTensorSliceTransfer_v3r2<const ck::Sequence<4, 16>, ck::tensor...",9773,23.119388,2.365639,0.500023
6304621,"ck::detail::TupleImpl<ck::Sequence<0, 1, 2, 3, 4, 5, 6, 7>, ck::Sequence<19>...",9773,23.119388,2.365639,0.500023
6304622,"ck::operator<<(std::ostream &, const TensorDescriptor<Tuple<Embed<Tuple<int,...",9773,23.119388,2.365639,0.500023
6304623,ck::TensorDescriptor<ck::Tuple<ck::Embed<ck::Tuple<ck::integral_constant<int...,9773,23.119388,2.365639,0.500023
6304624,"ck::sequence_gen<8, ck::lambda_get_up_dim_num<ck::Tuple<ck::PassThrough<int>...",9773,23.119388,2.365639,0.500023
6304625,std::unique_ptr<ck::tensor_operation::device::DeviceGroupedConvFwdMultipleAB...,9773,23.119388,2.365639,0.500023
6304626,"ck::container_concat<ck::Sequence<0>, ck::Sequence<2, 1>, ck::Sequence<3>, c...",9773,23.119388,2.365639,0.500023
6304627,"ck::Sequence<19, 20, 21, 22, 23>::ReorderGivenNew2Old<0, 1, 2, 3, 4>",9773,23.119388,2.365639,0.500023
6304628,"ck::make_tuple<ck::Sequence<0>, ck::Sequence<>, ck::Sequence<1>, ck::Sequenc...",9773,23.119388,2.365639,0.500023


### Template Depth Distribution

Analyze how template nesting affects compilation time.

In [16]:
if json_files and len(instantiations_df) > 0:
    # OPTIMIZATION: Aggregate by depth directly from instantiations, then join
    depth_stats = (
        instantiations_df.groupby("template_id")["dur_us"]
        .agg(["count", "sum", "mean", "median"])
        .reset_index()
    )

    # Join with templates to get depth
    depth_stats = depth_stats.merge(
        templates_df[["template_id", "depth"]], on="template_id"
    )

    # Now aggregate by depth
    depth_summary = (
        depth_stats.groupby("depth")
        .agg({"count": "sum", "sum": "sum", "mean": "mean", "median": "median"})
        .reset_index()
    )

    depth_summary.columns = ["depth", "count", "total_us", "mean_us", "median_us"]
    depth_summary["total_s"] = depth_summary["total_us"] / 1e6
    depth_summary["mean_ms"] = depth_summary["mean_us"] / 1e3
    depth_summary["median_ms"] = depth_summary["median_us"] / 1e3
    depth_summary["pct_total"] = (
        100 * depth_summary["total_us"] / total_template_time_us
    )

    print("Template Instantiation by Nesting Depth:")
    display(
        depth_summary[
            ["depth", "count", "total_s", "mean_ms", "median_ms", "pct_total"]
        ]
    )

Template Instantiation by Nesting Depth:


,depth,count,total_s,mean_ms,median_ms,pct_total
0,0,67779,3.204520e+02,5.207827,1.19350,0.266004
1,1,1749956922,7.353030e+06,5.587812,1.16350,6103.673137
2,2,3669923724,1.799595e+07,6.909845,1.26400,14938.245952
3,3,2019427209,1.064961e+07,6.790369,1.26300,8840.127967
4,4,2442047025,1.470272e+07,7.563676,1.27750,12204.571997
5,5,1644211492,1.041237e+07,8.437405,1.27700,8643.197486
6,6,756309412,6.665863e+06,15.362248,1.70650,5533.262772
7,7,59804841,5.608413e+05,15.135226,1.73000,465.548483
8,8,687425,6.183116e+03,14.874348,1.76475,5.132540


### Template Argument Analysis

Analyze template argument patterns to understand dependencies.


⚠️ **Warning** All the arg counts are 8192, join may be incorrect.

In [17]:
if json_files and len(template_args_df) > 0:
    # Count argument types
    arg_type_counts = template_args_df["arg_type"].value_counts()

    print("Template Argument Type Distribution:")
    print(f"{'Type':<15} {'Count':>10} {'Percentage':>12}")
    print("-" * 40)
    for arg_type, count in arg_type_counts.items():
        pct = 100 * count / len(template_args_df)
        print(f"{arg_type:<15} {count:>10,} {pct:>11.1f}%")

    # Templates with most arguments
    print("\nTemplates with Most Arguments:")
    arg_counts = (
        template_args_df.groupby("parent_template_id")
        .size()
        .reset_index(name="arg_count")
    )
    arg_counts = arg_counts.merge(
        templates_df[["template_id", "full_signature"]],
        left_on="parent_template_id",
        right_on="template_id",
    )
    display(arg_counts.nlargest(10, "arg_count")[["full_signature", "arg_count"]])

Template Argument Type Distribution:
Type                 Count   Percentage
----------------------------------------
template        116,601,482        80.8%
primitive       17,641,607        12.2%
unknown         10,098,716         7.0%

Templates with Most Arguments:


,full_signature,arg_count
223300,"ck::detail::TupleImpl<ck::Sequence<0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,...",33367
223301,"ck::vector_type<ck::bf8_fnuz_t, 8>::(unnamed union at /home/AMD/jshumway/com...",33367
223302,"ck::non_native_vector_base<ck::f8_fnuz_t, 64>::(unnamed union at /home/AMD/j...",33367
223303,"ck::Tuple<ck::f8_fnuz_t, ck::f8_fnuz_t, ck::f8_fnuz_t, ck::f8_fnuz_t, ck::f8...",33367
223304,"ck::Tuple<ck::non_native_vector_base<ck::f8_fnuz_t, 4>, ck::non_native_vecto...",33367
223305,"ck::Tuple<ck::non_native_vector_base<ck::f8_fnuz_t, 2>, ck::non_native_vecto...",33367
223306,"ck::non_native_vector_base<ck::f8_fnuz_t, 64>",33367
223307,"ck::Tuple<ck::f8_fnuz_t, ck::f8_fnuz_t, ck::f8_fnuz_t, ck::f8_fnuz_t, ck::f8...",33367
223308,"ck::Tuple<ck::f8_fnuz_t, ck::f8_fnuz_t, ck::f8_fnuz_t, ck::f8_fnuz_t, ck::f8...",33367
223309,"ck::Tuple<ck::non_native_vector_base<ck::f8_fnuz_t, 2>, ck::non_native_vecto...",33367


### Event Type Breakdown

In [18]:
if json_files and len(events_df) > 0:
    event_stats = (
        events_df.groupby("name", observed=True)
        .agg({"dur": ["count", "sum", "mean", "max"]})
        .reset_index()
    )
    event_stats.columns = ["event_type", "count", "total_us", "mean_us", "max_us"]
    event_stats["total_min"] = event_stats["total_us"] / 1e6 / 60
    event_stats["mean_ms"] = event_stats["mean_us"] / 1e3
    event_stats["pct_total"] = 100 * event_stats["total_us"] / total_build_time_us

    print("Top 20 Event Types by Total Duration:")
    display(
        event_stats.nlargest(20, "total_us")[
            ["event_type", "count", "total_min", "mean_ms", "pct_total"]
        ]
    )

Top 20 Event Types by Total Duration:


,event_type,count,total_min,mean_ms,pct_total
48,InstantiateFunction,13837259,1669.984159,7.241250,24.078188
31,ExecuteCompiler,1276,405.041858,19045.855415,5.839980
145,Total ExecuteCompiler,1276,405.041848,19045.854932,5.839980
149,Total Frontend,1276,348.010786,16364.143560,5.017694
33,Frontend,2462,348.010603,8481.168229,5.017691
47,InstantiateClass,1969340,337.831372,10.292729,4.870925
243,Total Source,1276,220.549562,10370.669049,3.179931
170,Total InstantiateFunction,1276,154.873240,7282.440762,2.232995
78,PerformPendingInstantiations,1276,94.353463,4436.683223,1.360408
219,Total PerformPendingInstantiations,1276,94.353453,4436.682719,1.360408


## Part 4: Build Parallelism Analysis

Analyze build-level parallelism using ninja's `.ninja_log` file to understand how well the build system utilizes available CPU cores.

### Parse Ninja Build Log


⚠️ **Warning** The `worker_id` values are all -1. We need to verify this data and sanity-check the timestamps. 


In [19]:
from trace_analysis import NinjaLogParser

# Look for .ninja_log
ninja_log_paths = [
    Path("../../../build-trace/.ninja_log"),
]

ninja_log = None
for path in ninja_log_paths:
    if path.exists():
        ninja_log = path
        break

if ninja_log:
    print(f"Found ninja log: {ninja_log}")

    # Parse ninja log
    start = time.time()
    builds = NinjaLogParser.parse(ninja_log)
    elapsed = time.time() - start

    print(f"Parsed {len(builds):,} build events in {elapsed:.3f}s")

    # Convert to DataFrame
    builds_df = NinjaLogParser.to_dataframe(builds)
    print(f"\nBuilds DataFrame: {len(builds_df):,} rows")
    display(builds_df.head())
else:
    print("No .ninja_log found. Run a ninja build first.")

Found ninja log: ../../../build-trace/.ninja_log
Parsed 2,536 build events in 0.003s

Builds DataFrame: 2,536 rows


,target,start_ms,end_ms,cmd_hash,worker_id,duration_ms
0,/home/AMD/jshumway/composable_kernel/build-trace/CMakeFiles/cmake.verify_globs,487,511,20b4a9dba5446149,-1,24
1,build.ninja,512,159966,80f2b673d9d1b995,-1,159454
2,library/src/tensor_operation_instance/gpu/grouped_conv2d_fwd/CMakeFiles/devi...,354,9957,b37d14fd75e2d29c,-1,9603
3,_deps/gtest-build/googletest/CMakeFiles/gtest.dir/src/gtest-all.cc.o,225,9931,330bec9abe978cde,-1,9706
4,library/src/tensor_operation_instance/gpu/grouped_conv2d_fwd/CMakeFiles/devi...,365,229598,6a5b0a35a99473e8,-1,229233


## Conclusion

This notebook demonstrated how to:

1. **Parse** Clang `-ftime-trace` files efficiently using parallel processing
2. **Transform** raw JSON into structured multi-table schemas
3. **Analyze** build performance using relational queries
4. **Visualize** build parallelism with Gantt charts
5. **Identify** optimization opportunities with data-driven techniques

### Key Advantages of Enhanced Schema

- **Memory efficient**: Normalized tables reduce redundancy
- **Query performance**: Optimized dtypes and indexes
- **Relational analysis**: Join tables to explore dependencies
- **Scalable**: Handles large builds with minimal memory

### Performance Optimizations

- **Aggregate first, join later**: Reduces dataset size before expensive joins
- **Optimized dtypes**: int32, int8, category types minimize memory
- **Parallel processing**: Utilizes all CPU cores for parsing
- **Efficient queries**: Pandas operations optimized for large datasets

### Build Parallelism Insights

TODO: See if we can recreate Perfetto view of the full ninja build.

### Next Steps

- Customize queries for your specific analysis needs
- Track build time trends over time
- Correlate template complexity with build times
- Optimize critical path builds
- Share insights with your team

### Resources

- [Clang -ftime-trace Documentation](https://releases.llvm.org/11.0.0/tools/clang/docs/ClangCommandLineReference.html#cmdoption-clang-ftime-trace)
- [Chrome Trace Event Format](https://docs.google.com/document/d/1CvAClvFfyA5R-PhYUmn5OOQtYMH4h6I0nSsKchNAySU/preview)
- [Ninja Build System](https://ninja-build.org/)
- [trace_analysis Library Documentation](../trace_analysis/README.md)